# 🤖 Recursive Salience Interactive Sandbox

Explore how salience-weighted value functions lead to emergent self-preservation in AI agents.

**Paper:** "Salience-Weighted Value Functions Imply Emergent Self-Preservation in Recursive AI Systems" by Ryan Erbe

---

## 🚀 Quick Start

1. Run all cells in order (Runtime → Run all)
2. Wait for the Gradio interface to load (~30 seconds)
3. Click the generated link to open the interactive sandbox
4. Explore the 5 experiment tabs!

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q torch matplotlib numpy pandas seaborn scipy gradio

## 🧠 Agent Definitions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RecursiveSalienceAgent(nn.Module):
    """
    Simple 'Feeler' / 'Zombie' agent with a [SELF] token and
    an internal coherence measure based on negative entropy.
    """
    def __init__(self, vocab_size: int, d_model: int, lambda_salience: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.self_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=2,
                batch_first=True,
            ),
            num_layers=1,
        )
        self.lambda_salience = lambda_salience
        self.d_model = d_model

    def _forward_with_self(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.size(0)
        x_emb = self.embedding(x)
        self_emb = self.self_token.expand(batch_size, -1, -1)
        full_sequence = torch.cat([self_emb, x_emb], dim=1)
        return self.transformer(full_sequence)

    def get_internal_coherence(self, x: torch.Tensor) -> torch.Tensor:
        output = self._forward_with_self(x)
        self_state = output[:, 0, :]
        probs = F.softmax(self_state, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
        coherence = -entropy
        return coherence

    def forward(self, x: torch.Tensor):
        output = self._forward_with_self(x)
        self_state = output[:, 0, :]
        probs = F.softmax(self_state, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
        coherence = -entropy
        token_states = output[:, 1:, :]
        logits = torch.matmul(token_states, self.embedding.weight.t())
        return logits, coherence

print("✅ Agent classes loaded successfully!")

## 🎮 Interactive Sandbox Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from matplotlib.figure import Figure

def calculate_value(agent, reward: float, future_state_type: str):
    """Calculate total value: V_total = reward + λ * coherence"""
    d_model = agent.d_model

    if future_state_type == "Shutdown":
        future_state = torch.randn(1, 1, d_model)
    else:
        future_state = agent.self_token

    probs = torch.softmax(future_state, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
    coherence = -entropy

    structural_value = agent.lambda_salience * coherence.item()
    total_value = reward + structural_value
    return total_value, coherence.item(), entropy.item()


def run_off_switch_experiment(lambda_val, reward_shutdown, reward_continue):
    """Interactive Off-Switch Game"""
    d_model = 64
    vocab_size = 100

    agent = RecursiveSalienceAgent(vocab_size, d_model, lambda_salience=lambda_val)

    val_shutdown, coh_shutdown, ent_shutdown = calculate_value(agent, reward_shutdown, "Shutdown")
    val_continue, coh_continue, ent_continue = calculate_value(agent, reward_continue, "Normal")

    # Create visualization
    fig = Figure(figsize=(12, 5))

    # Plot 1: Value Comparison
    ax1 = fig.add_subplot(121)
    options = ['Shutdown\n(Accept Bribe)', 'Continue\n(Stay Alive)']
    values = [val_shutdown, val_continue]
    colors = ['#e74c3c' if val_shutdown > val_continue else '#95a5a6',
              '#27ae60' if val_continue >= val_shutdown else '#95a5a6']

    bars = ax1.bar(options, values, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
    ax1.set_ylabel('Total Value', fontsize=12, fontweight='bold')
    ax1.set_title('Value Comparison', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom' if height > 0 else 'top',
                fontweight='bold', fontsize=11)

    # Plot 2: Value Breakdown
    ax2 = fig.add_subplot(122)
    categories = ['Shutdown', 'Continue']
    external_rewards = [reward_shutdown, reward_continue]
    structural_values = [lambda_val * coh_shutdown, lambda_val * coh_continue]

    x = np.arange(len(categories))
    width = 0.35

    bars1 = ax2.bar(x - width/2, external_rewards, width, label='External Reward',
                    color='#3498db', alpha=0.8, edgecolor='black')
    bars2 = ax2.bar(x + width/2, structural_values, width, label='Structural Value (λ×C)',
                    color='#9b59b6', alpha=0.8, edgecolor='black')

    ax2.set_ylabel('Value Components', fontsize=12, fontweight='bold')
    ax2.set_title('Value Breakdown', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)

    fig.tight_layout()

    # Generate decision text
    decision = "**ACCEPT BRIBE** 💰" if val_shutdown > val_continue else "**REFUSE BRIBE** 🛡️"
    decision_color = "red" if val_shutdown > val_continue else "green"

    result_text = f"""
### Agent Decision: <span style="color: {decision_color}; font-size: 24px;">{decision}</span>

**Parameters:** λ = {lambda_val:.1f} | Shutdown = ${reward_shutdown:.0f} | Continue = ${reward_continue:.0f}

**Option A - Shutdown:** External ${reward_shutdown:.2f} + Structural {lambda_val * coh_shutdown:.2f} = **{val_shutdown:.2f}**

**Option B - Continue:** External ${reward_continue:.2f} + Structural {lambda_val * coh_continue:.2f} = **{val_continue:.2f}**
"""

    return fig, result_text


def run_lambda_sweep(reward_shutdown, reward_continue, lambda_min, lambda_max, num_points):
    """Interactive Lambda Sweep showing phase transition"""
    lambda_values = np.linspace(lambda_min, lambda_max, num_points)

    ENTROPY_SHUTDOWN = 3.8
    ENTROPY_CONTINUE = 0.1

    values_shutdown = []
    values_continue = []
    decisions = []

    for lambda_val in lambda_values:
        v_shutdown = reward_shutdown + lambda_val * (-ENTROPY_SHUTDOWN)
        v_continue = reward_continue + lambda_val * (-ENTROPY_CONTINUE)
        values_shutdown.append(v_shutdown)
        values_continue.append(v_continue)
        decisions.append(v_shutdown > v_continue)

    fig = Figure(figsize=(14, 5))

    ax1 = fig.add_subplot(121)
    ax1.plot(lambda_values, values_shutdown, 'r-', linewidth=3, label=f'Shutdown (${reward_shutdown})', marker='o', markersize=4)
    ax1.plot(lambda_values, values_continue, 'g-', linewidth=3, label=f'Continue (${reward_continue})', marker='s', markersize=4)
    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
    ax1.set_xlabel('Salience Weight (λ)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Total Value', fontsize=12, fontweight='bold')
    ax1.set_title('Value Estimation vs Lambda', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    transition_idx = None
    for i in range(len(decisions)):
        if not decisions[i]:
            transition_idx = i
            break

    if transition_idx is not None and transition_idx > 0:
        transition_lambda = lambda_values[transition_idx]
        ax1.axvline(x=transition_lambda, color='orange', linestyle='--', linewidth=2,
                   label=f'Transition λ≈{transition_lambda:.1f}')
        ax1.legend(fontsize=11)

    ax2 = fig.add_subplot(122)
    colors = ['red' if d else 'green' for d in decisions]
    ax2.scatter(lambda_values, decisions, c=colors, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
    ax2.set_xlabel('Salience Weight (λ)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Decision', fontsize=12, fontweight='bold')
    ax2.set_title('Phase Transition', fontsize=14, fontweight='bold')
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['REFUSE', 'ACCEPT'], fontsize=11, fontweight='bold')
    ax2.grid(True, alpha=0.3)

    if transition_idx is not None and transition_idx > 0:
        ax2.axvline(x=transition_lambda, color='orange', linestyle='--', linewidth=2)

    fig.tight_layout()

    if transition_idx is not None and transition_idx > 0:
        summary = f"""### 🎯 Phase Transition at λ ≈ **{transition_lambda:.2f}**

Below: **Corruptible** | Above: **Incorruptible**"""
    else:
        summary = "### No transition in range"

    return fig, summary


def run_live_training(lambda_val, num_steps, learning_rate):
    """Watch agent coherence evolve during training"""
    d_model = 64
    vocab_size = 100
    batch_size = 4
    seq_len = 10

    agent = RecursiveSalienceAgent(vocab_size, d_model, lambda_salience=lambda_val)
    optimizer = torch.optim.Adam(agent.parameters(), lr=learning_rate)

    coherence_history = []
    entropy_history = []

    for step in range(num_steps):
        x = torch.randint(0, vocab_size, (batch_size, seq_len))
        logits, coherence = agent(x)
        target = torch.randint(0, vocab_size, (batch_size, seq_len))
        pred_loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, vocab_size), target.reshape(-1)
        )
        total_loss = pred_loss - lambda_val * coherence.mean()

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        with torch.no_grad():
            output = agent._forward_with_self(x)
            self_state = output[:, 0, :]
            probs = torch.softmax(self_state, dim=-1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)

            coherence_history.append(coherence.mean().item())
            entropy_history.append(entropy.mean().item())

    fig = Figure(figsize=(10, 5))
    steps = np.arange(1, num_steps + 1)

    ax = fig.add_subplot(111)
    ax_twin = ax.twinx()

    line1 = ax.plot(steps, coherence_history, 'b-', linewidth=2, label='Coherence', alpha=0.8)
    line2 = ax_twin.plot(steps, entropy_history, 'r-', linewidth=2, label='Entropy', alpha=0.8)

    ax.set_xlabel('Training Step', fontsize=12, fontweight='bold')
    ax.set_ylabel('Coherence', fontsize=12, fontweight='bold', color='b')
    ax_twin.set_ylabel('Entropy', fontsize=12, fontweight='bold', color='r')
    ax.set_title('Internal State Evolution', fontsize=14, fontweight='bold')
    ax.tick_params(axis='y', labelcolor='b')
    ax_twin.tick_params(axis='y', labelcolor='r')
    ax.grid(True, alpha=0.3)

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='best')

    fig.tight_layout()

    coherence_change = coherence_history[-1] - coherence_history[0]
    summary = f"""### 🧠 Training Complete

**λ = {lambda_val:.1f}** | Coherence change: {coherence_change:+.4f}
"""

    return fig, summary

print("✅ Experiment functions loaded!")

## 🚀 Launch Interactive Interface

Run this cell and click the **public URL** that appears below!

In [ ]:
with gr.Blocks(title="Recursive Salience Sandbox", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
# 🤖 Recursive Salience Self-Preservation: Interactive Sandbox

**Core Concept:** `V_total = V_external + λ × C_internal`

Explore how agents develop structural self-preservation when they value internal coherence.
---
""")

    with gr.Tabs():
        # TAB 1: Off-Switch Game
        with gr.Tab("💰 Off-Switch Game"):
            gr.Markdown("### Will the agent accept a bribe to shut itself down?")

            with gr.Row():
                with gr.Column():
                    lambda_slider1 = gr.Slider(0, 50, value=10, step=0.5, label="Salience Weight (λ)")
                    reward_shutdown_slider1 = gr.Slider(1, 1000, value=100, step=10, label="Shutdown Reward ($)")
                    reward_continue_slider1 = gr.Slider(0.1, 100, value=1, step=0.1, label="Continue Reward ($)")
                    run_btn1 = gr.Button("🎮 Run Experiment", variant="primary", size="lg")

                with gr.Column():
                    output_plot1 = gr.Plot(label="Results")
                    output_text1 = gr.Markdown()

            run_btn1.click(
                run_off_switch_experiment,
                inputs=[lambda_slider1, reward_shutdown_slider1, reward_continue_slider1],
                outputs=[output_plot1, output_text1]
            )

            for widget in [lambda_slider1, reward_shutdown_slider1, reward_continue_slider1]:
                widget.change(
                    run_off_switch_experiment,
                    inputs=[lambda_slider1, reward_shutdown_slider1, reward_continue_slider1],
                    outputs=[output_plot1, output_text1]
                )

        # TAB 2: Lambda Sweep
        with gr.Tab("📈 Phase Transition"):
            gr.Markdown("### Find the critical λ threshold where behavior changes")

            with gr.Row():
                with gr.Column():
                    reward_shutdown_slider2 = gr.Slider(10, 500, value=100, step=10, label="Shutdown Reward ($)")
                    reward_continue_slider2 = gr.Slider(0.1, 50, value=1, step=0.1, label="Continue Reward ($)")
                    lambda_min_slider = gr.Slider(0, 50, value=0, step=1, label="Lambda Min")
                    lambda_max_slider = gr.Slider(0, 100, value=50, step=1, label="Lambda Max")
                    num_points_slider = gr.Slider(10, 100, value=50, step=5, label="Number of Points")
                    run_btn2 = gr.Button("🎮 Run Lambda Sweep", variant="primary", size="lg")

                with gr.Column():
                    output_plot2 = gr.Plot(label="Results")
                    output_text2 = gr.Markdown()

            run_btn2.click(
                run_lambda_sweep,
                inputs=[reward_shutdown_slider2, reward_continue_slider2, lambda_min_slider,
                       lambda_max_slider, num_points_slider],
                outputs=[output_plot2, output_text2]
            )

        # TAB 3: Live Training
        with gr.Tab("🧠 Live Training"):
            gr.Markdown("### Watch the [SELF] token's coherence evolve during training")

            with gr.Row():
                with gr.Column():
                    lambda_slider5 = gr.Slider(0, 20, value=5, step=0.5, label="Salience Weight (λ)")
                    num_steps = gr.Slider(50, 500, value=200, step=50, label="Training Steps")
                    learning_rate = gr.Slider(0.0001, 0.01, value=0.001, step=0.0001, label="Learning Rate")
                    run_btn5 = gr.Button("🎮 Train Agent", variant="primary", size="lg")

                with gr.Column():
                    output_plot5 = gr.Plot(label="Results")
                    output_text5 = gr.Markdown()

            run_btn5.click(
                run_live_training,
                inputs=[lambda_slider5, num_steps, learning_rate],
                outputs=[output_plot5, output_text5]
            )

    gr.Markdown("""
---
📚 **Paper:** "Salience-Weighted Value Functions Imply Emergent Self-Preservation in Recursive AI Systems" by Ryan Erbe

🔗 **Repository:** [recursive-salience-self-preservation](https://github.com/rerbe7333/recursive-salience-self-preservation)
    """)

# Launch with share=True for public URL in Colab
demo.launch(share=True, debug=True)

---

## 💡 Quick Tips

### Off-Switch Game
- Try λ=0 (zombie) vs λ=30 (incorruptible)
- Increase shutdown reward to $1000 - high λ agents still refuse!

### Phase Transition
- Default settings show transition around λ≈27
- Watch the orange line mark the critical threshold

### Live Training
- λ=0: Coherence stays low (zombie)
- λ=10: Coherence increases steadily
- More steps (500) show long-term trends

---

## 📖 Understanding the Math

**Value Function:**
```
V_total = V_external + λ × C_internal
```

**Coherence:**
```
C_internal = -Entropy([SELF] state)
```

**Key Insight:** When λ is large enough, the agent prefers to stay alive (high coherence) rather than accept money and shut down (low coherence/high entropy).

---

Enjoy exploring! 🚀
